# **Section 1: CNV Data Exploration, Cleaning, and Preprocessing**

This notebook performs comprehensive data exploration and cleaning to prepare CNV depth data for machine learning.
Unlike the previous workflow, feature selection is intentionally **excluded** from this notebook to **avoid data leakage**.
Feature selection will be performed within cross-validation folds during model training.

**Contents:**
0. Helper functions for parsing resistance profiles
1. Load and preprocess data
2. Clean data (remove duplicates, handle NaN values)
3. Exploratory analysis and visualization
4. Save preprocessed data

**Important:** Feature selection has been moved to the model training pipeline to prevent data leakage.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
sns.set_palette('pastel')

## **0. Helper Functions**

### **0.1 Parse Resistance Profiles**

The function `parse_resistance_profile` extracts resistance phenotypes from the genome_metadata.csv file.

In [ ]:
def parse_resistance_profile(profile_str):
    """
    Parse resistance profile string into individual antibiotic phenotypes.

    Format: "Resistant to X, Y; Susceptible to A, B; Intermediate to C; Not determined: D"
    Returns: dict with antibiotic: phenotype (R/S/I/ND)
    """
    phenotypes = {}
    
    if pd.isna(profile_str):
        return phenotypes
    
    # Split by semicolon to get phenotype groups
    parts = str(profile_str).split(';')
    
    for part in parts:
        part = part.strip()
        if not part:
            continue
            
        # Determine phenotype
        if part.startswith('Resistant to'):
            phenotype = 'R'
            antibiotics_str = part.replace('Resistant to', '').strip()
        elif part.startswith('Susceptible to'):
            phenotype = 'S'
            antibiotics_str = part.replace('Susceptible to', '').strip()
        elif part.startswith('Intermediate to'):
            phenotype = 'I'
            antibiotics_str = part.replace('Intermediate to', '').strip()
        elif part.startswith('Not determined'):
            phenotype = 'ND'
            antibiotics_str = part.replace('Not determined:', '').strip()
        else:
            continue
        
        # Split antibiotics by comma and add to dict
        antibiotics = [ab.strip() for ab in antibiotics_str.split(',')]
        for antibiotic in antibiotics:
            if antibiotic:
                phenotypes[antibiotic] = phenotype
    
    return phenotypes

## **1. Load and Preprocess Data**

### **1.1 Load CNV Feature Matrix**

In [ ]:
# Load raw CNV depth matrix
cnv_matrix = pd.read_csv('Data/cnv_depth_matrix.tsv', sep='\t', index_col=0)

print(f'Feature Matrix Shape: {cnv_matrix.shape}')
print(f'Data types: {cnv_matrix.dtypes.unique()}')
print(f'Missing values (NaN): {cnv_matrix.isna().sum().sum()} / {cnv_matrix.size} ({100*cnv_matrix.isna().sum().sum()/cnv_matrix.size:.2f}%)')

### **1.2 Load Resistance Profiles**

In [ ]:
# Load metadata with resistance profiles
metadata = pd.read_csv('../genome_metadata.csv')

print(f'Metadata shape: {metadata.shape}')
print(f'Columns: {metadata.columns.tolist()}')
print(f'\nFirst few rows:')
print(metadata.head())

### **1.3 Parse Resistance Profiles**

In [ ]:
# Parse resistance profiles into antibiotic-phenotype dictionaries
parsed_resistances = metadata['resistance_profile'].apply(parse_resistance_profile)

# Convert to DataFrame
resistances_df = pd.json_normalize(parsed_resistances)

# Add sample identifier (SRA accession) and set as index
resistances_df['sra_accession'] = metadata['sra_accession'].values
resistances_df = resistances_df.set_index('sra_accession')

print(f'Resistance profiles shape: {resistances_df.shape}')
print(f'Antibiotics detected: {resistances_df.columns.tolist()}')
print(f'\nFirst few rows:')
print(resistances_df.head())

### **1.4 Find Common Samples**

In [ ]:
# Remove duplicate entries in both dataframes
resistances_df = resistances_df.loc[~resistances_df.index.duplicated(keep='first')]
cnv_matrix = cnv_matrix.loc[~cnv_matrix.index.duplicated(keep='first')]

print(f'Before filtering:')
print(f'  CNV samples: {len(cnv_matrix)}')
print(f'  Samples with resistance profiles: {len(resistances_df)}')

# Keep only common samples
common_samples = cnv_matrix.index.intersection(resistances_df.index)
cnv_matrix = cnv_matrix.loc[common_samples]
resistances_df = resistances_df.loc[common_samples]

print(f'\nAfter filtering:')
print(f'  Common samples: {len(cnv_matrix)}')
print(f'  CNV features: {cnv_matrix.shape[1]}')
print(f'  Resistance profiles: {resistances_df.shape[1]}')

### **1.5 Filter by SXT Resistance Profile**

In [ ]:
# Extract SXT resistance profile
SXT_profile = resistances_df['SXT'].copy()

# Keep only samples with valid SXT phenotype (S or R)
valid_samples = SXT_profile[SXT_profile.isin(['S', 'R'])].index
cnv_matrix = cnv_matrix.loc[valid_samples]
SXT_profile = SXT_profile.loc[valid_samples]

# Convert to binary (0=Susceptible, 1=Resistant)
SXT_binary = (SXT_profile == 'R').astype(int)

print(f'Valid samples with SXT phenotype: {len(cnv_matrix)}')
print(f'Features: {cnv_matrix.shape[1]}')
print(f'\nSXT resistance distribution:')
print(SXT_binary.value_counts())
print(f'  Resistant (1): {(SXT_binary==1).sum()}')
print(f'  Susceptible (0): {(SXT_binary==0).sum()}')

## **2. Clean Data**

### **2.1 Remove Duplicate Samples**

In [ ]:
# Identify duplicate samples based on CNV values
duplicate_samples = cnv_matrix.duplicated(keep='first')

print(f'Duplicate samples: {duplicate_samples.sum()}')

if duplicate_samples.sum() > 0:
    print(f'Duplicate sample indices: {cnv_matrix.index[duplicate_samples].tolist()}')
    
    # Remove duplicates
    cnv_matrix = cnv_matrix.loc[~duplicate_samples]
    SXT_binary = SXT_binary.loc[~duplicate_samples]
    
    print(f'\nAfter removing duplicates:')
    print(f'  Samples: {len(cnv_matrix)}')
else:
    print('No duplicate samples found')

### **2.2 Remove Duplicate Features**

In [ ]:
# Identify duplicate features (identical across all samples)
cnv_T = cnv_matrix.T
duplicate_features = cnv_T.duplicated(keep='first')

print(f'Duplicate features: {duplicate_features.sum()}')
print(f'Features before: {cnv_matrix.shape[1]}')

# Remove duplicates
cnv_matrix = cnv_matrix.loc[:, ~duplicate_features]

print(f'Features after: {cnv_matrix.shape[1]}')
print(f'Removed: {duplicate_features.sum()} duplicate features')

### **2.3 Fill NaN Values**

In [ ]:
# Report NaN values before filling
print(f'NaN values before filling: {cnv_matrix.isna().sum().sum()} / {cnv_matrix.size}')
print(f'Percentage: {100*cnv_matrix.isna().sum().sum()/cnv_matrix.size:.2f}%')

# Fill NaN with -20 (represents absent/unexpressed genes)
# Since log2(CNV) should be ~0 for normal copy number, -20 represents very low/absent
cnv_matrix.fillna(-20, inplace=True)

print(f'\nNaN values after filling: {cnv_matrix.isna().sum().sum()}')
print(f'\nFinal dataset shape: {cnv_matrix.shape}')

## **3. Exploratory Analysis and Visualization**

### **3.1 Per-Sample Summary Statistics**

In [ ]:
# Calculate per-sample statistics
sample_stats = pd.DataFrame({
    'sample': cnv_matrix.index,
    'mean': cnv_matrix.mean(axis=1).values,
    'median': cnv_matrix.median(axis=1).values,
    'std': cnv_matrix.std(axis=1).values,
    'min': cnv_matrix.min(axis=1).values,
    'max': cnv_matrix.max(axis=1).values,
    'q25': cnv_matrix.quantile(0.25, axis=1).values,
    'q75': cnv_matrix.quantile(0.75, axis=1).values
})

sample_stats['iqr'] = sample_stats['q75'] - sample_stats['q25']
sample_stats = sample_stats.set_index('sample')

print('Sample Statistics Summary:')
print(f'  Mean CNV (across samples): {sample_stats["mean"].mean():.4f}')
print(f'  Max value: {sample_stats["max"].max():.4f}')
print(f'  Min value: {sample_stats["min"].min():.4f}')
print(f'  Median std dev: {sample_stats["std"].median():.4f}')

### **3.2 Visualize Sample Statistics**

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Mean distribution
axes[0, 0].hist(sample_stats['mean'], bins=50, alpha=0.7, edgecolor='black')
axes[0, 0].set_xlabel('Mean CNV (log2 ratio)', fontweight='bold')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of Mean CNVs', fontweight='bold')
axes[0, 0].grid(alpha=0.3)

# Standard deviation distribution
axes[0, 1].hist(sample_stats['std'], bins=50, alpha=0.7, edgecolor='black')
axes[0, 1].set_xlabel('CNV Standard Deviation', fontweight='bold')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Distribution of CNV Std Dev', fontweight='bold')
axes[0, 1].grid(alpha=0.3)

# Max value distribution
axes[1, 0].hist(sample_stats['max'], bins=50, alpha=0.7, edgecolor='black')
axes[1, 0].set_xlabel('Max CNV (log2 ratio)', fontweight='bold')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Max CNV Per Sample', fontweight='bold')
axes[1, 0].grid(alpha=0.3)

# Min value distribution
axes[1, 1].hist(sample_stats['min'], bins=50, alpha=0.7, edgecolor='black')
axes[1, 1].set_xlabel('Min CNV (log2 ratio)', fontweight='bold')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Min CNV Per Sample', fontweight='bold')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('Figures/01_statistics.png', dpi=300, bbox_inches='tight')
plt.show()

### **3.3 Boxplot of CNV Values Per Sample**

In [ ]:
# Sort samples by mean CNV for better visualization
sorted_samples = sample_stats.sort_values('mean', ascending=False).index
cnv_sorted = cnv_matrix.loc[sorted_samples]

fig, ax = plt.subplots(figsize=(16, 8))
sns.boxplot(cnv_sorted.T.values, showfliers=False)
ax.set_xlabel('Sample', fontweight='bold')
ax.set_ylabel('CNV log2 Ratio', fontweight='bold')
ax.set_title('CNV Value Distribution Per Sample', fontweight='bold', fontsize=13)
plt.xticks(rotation=90, fontsize=8)
plt.tight_layout()
plt.savefig('Figures/01_boxplot_per_sample.png', dpi=300, bbox_inches='tight')
plt.show()

### **3.4 Feature Statistics**

In [ ]:
# Calculate per-feature statistics
feature_stats = pd.DataFrame({
    'mean': cnv_matrix.mean(axis=0),
    'std': cnv_matrix.std(axis=0),
    'min': cnv_matrix.min(axis=0),
    'max': cnv_matrix.max(axis=0),
    'variance': cnv_matrix.var(axis=0)
})

print('Feature Statistics Summary:')
print(f'  Total features: {len(feature_stats)}')
print(f'  Features with zero variance: {(feature_stats["variance"] == 0).sum()}')
print(f'  Mean feature variance: {feature_stats["variance"].mean():.4f}')
print(f'  Median feature variance: {feature_stats["variance"].median():.4f}')

## **4. Save Preprocessed Data**

In [ ]:
# Save cleaned CNV matrix
cnv_matrix.to_csv('Data/cnvs_cleaned.csv')
print(f'Saved: Data/cnvs_cleaned.csv ({cnv_matrix.shape})')

# Save SXT resistance profile
SXT_binary.to_csv('Data/SXT_cleaned.csv', header=['SXT_binary'])
print(f'Saved: Data/SXT_cleaned.csv ({len(SXT_binary)} samples)')

print(f'\n\nPreprocessing Summary:')
print(f'  Samples: {cnv_matrix.shape[0]}')
print(f'  Features: {cnv_matrix.shape[1]}')
print(f'  Resistant: {(SXT_binary==1).sum()}')
print(f'  Susceptible: {(SXT_binary==0).sum()}')
print(f'\nNote: Feature selection will be performed within cross-validation folds during model training to avoid data leakage.')